In [ ]:
import pandas as pd
from datetime import datetime
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
%load_ext autoreload
%autoreload 2

## Metrics results class

### Metrics types

In [ ]:
from pulsar_metrics.metrics.enums import MetricsType

In [ ]:
MetricsType._member_names_

### Metric data structure

In [ ]:
from pulsar_metrics.metrics.base import MetricResults

In [ ]:
results = MetricResults(
    metric_type = 'custom', metric_name = 'accuracy')
results

## Loading datasets

In [ ]:
data_ref = pd.read_csv('data/california_ref.csv')
data_new = pd.read_csv('data/california_new.csv')

In [ ]:
data_ref.head()

## Summary statistics

In [ ]:
from pulsar_metrics.metrics.statistics import FeatureSummary

In [ ]:
statistics = FeatureSummary(feature_name='HouseAge')

In [ ]:
statistics.evaluate(current = data_new, reference=data_ref)

In [ ]:
statistics.results_to_pandas()

## Drift Class

In [ ]:
from pulsar_metrics.metrics.drift import DriftTestMetric, DriftMetric
from pulsar_metrics.metrics.enums import DriftMetricsFuncs, DriftTestMetricsFuncs

In [ ]:
DriftMetricsFuncs._member_names_

In [ ]:
DriftTestMetricsFuncs._member_names_

### Test Drift metrics

In [ ]:
feature_name = 'MedInc'
#feature_name = 'Population'
fig, ax = plt.subplots(1, 1, figsize = (10, 8))
data_ref[feature_name].plot.density(ax = ax, alpha = 0.5, label = 'reference')
plt.axvline(data_ref[feature_name].mean())
data_new[feature_name].plot.density(ax = ax, alpha = 0.5, label = 'New')
plt.axvline(data_new[feature_name].mean())
#ax.set_xlim((0, 6000))
plt.legend()

In [ ]:

driftTest = DriftTestMetric(metric_name = 'ttest', feature_name = feature_name)
driftTest.evaluate(alpha = 0.05, current = data_new, reference = data_ref)

In [ ]:
driftTest = DriftTestMetric(metric_name = 'CvM', feature_name = feature_name)
driftTest.evaluate(alpha = 0.05, current = data_new, reference = data_ref)

### Non test drift metrics

In [ ]:
drift = DriftMetric(metric_name = 'wasserstein', feature_name = feature_name)
drift.evaluate(current = data_new, reference = data_ref, threshold = 0.2, upper_bound = True)

In [ ]:
drift = DriftMetric(metric_name = 'mmd', feature_name = feature_name)
drift.evaluate(current = data_new, reference = data_ref, threshold = 0.2, upper_bound = True, kernel='rbf')

## Custom metric class    

In [ ]:
from pulsar_metrics.metrics.drift import CustomDriftMetric

In [ ]:
@CustomDriftMetric
def test_custom(current, reference, multiple=3, **kwargs):
    return multiple*np.max(current - reference)

In [ ]:
cus = test_custom(metric_name = 'test', feature_name = feature_name)

In [ ]:
cus.evaluate(current = data_new, reference = data_ref, threshold = 1, multiple=0.5)

## Performance metrics

### Implemented metrics

In [ ]:
from pulsar_metrics.metrics.enums import PerformanceMetricsFuncs

In [ ]:
PerformanceMetricsFuncs._member_names_

### Exemple performance metric

In [ ]:
from pulsar_metrics.metrics.performance import PerformanceMetric

In [ ]:
precision = PerformanceMetric(metric_name = 'precision', y_name = 'clf_target')

In [ ]:
precision.evaluate(current = data_new, bootstrap=True, average = 'micro', threshold = [0.45, 0.5])

## Running an analyzer with multiple metrics

In [ ]:
from pulsar_metrics.analyzers.base import Analyzer

In [ ]:
analysis = Analyzer(name = 'First Analyzer', description='My first Analyzer', model_id = 1, model_version=2)

In [ ]:
analysis.add_drift_metrics(metrics_list=['wasserstein', 'ttest', 'ks_2samp', 'mmd'], features_list=['AveBedrms', 'MedInc']);
analysis.add_performance_metrics(metrics_list=['accuracy'], y_name = 'clf_target');

In [ ]:
analysis.run(reference = data_ref, current = data_new, options = {'ttest': {'alpha': 0.01, 'equal_var': False}, 'mmd': {'kernel': 'rbf'}})

In [ ]:
analysis._metadata

In [ ]:
results = analysis.results_to_pandas()
results

In [ ]:
results.loc[(results.metric_type=='statistics') & (results.metric_name=='count')]